# Ensemble: BERTimbau Large (múltiplas seeds) + SVM TF-IDF

Este notebook combina:
1. **Três BERTimbau Large**, treinados independentemente com seeds diferentes.
2. **Um SVM linear com TF-IDF**, otimizado por validação cruzada estratificada.
3. **Fusão tardia (late fusion)**: combinação das probabilidades dos BERTimbau com as probabilidades calibradas do SVM.

O objetivo de seleção é **F1-Macro**, pois considera igualmente as três classes (`c1`, `c234`, `c5`). A acurácia também será reportada.

## Protocolo experimental e cuidados contra vazamento

- `train.xlsx` é dividido uma única vez em treino e validação estratificados.
- O Grid Search do SVM ocorre **somente dentro do conjunto de treino**, por 5-fold CV.
- Os BERTimbau são selecionados pela perda/F1 na validação, com early stopping.
- A validação serve para comparar os modelos e escolher o peso da fusão. Portanto, os resultados nela são resultados de seleção, não uma estimativa totalmente independente.
- Depois da seleção, os modelos finais são reajustados usando **todo o `train.xlsx`** e geram previsões para `test2.xlsx`.
- `test2.xlsx` não é usado para escolher hiperparâmetros, pesos ou épocas.

> **Custo computacional:** BERTimbau Large é pesado. Três seeds com até 4 épocas podem levar bastante tempo em uma T4. Comece com `N_SEEDS=3`; se faltar tempo ou VRAM, use `N_SEEDS=2`, batch menor ou `MAX_LEN=192`. Não aumente a quantidade de modelos sem comparar o ganho real.


## 1. Instalação e imports

Execute esta célula no Colab com GPU habilitada: **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU**.

In [1]:
!pip -q install -U transformers accelerate scikit-learn openpyxl

In [2]:
import os
import re
import gc
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Ative uma GPU no Colab antes de treinar o BERTimbau.")

PyTorch: 2.14.1+cu130
CUDA disponível: True
GPU: NVIDIA A100-SXM4-40GB


## 2. Drive, caminhos, configuração e reprodutibilidade

Ajuste `BASE_DIR` para a pasta onde estão `train.xlsx` e, se disponível, `test2.xlsx`. O código mantém o mapeamento de classes usado no EP2.

In [3]:
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/epPLN/ep2")

TRAIN_PATH = BASE_DIR / "train.xlsx"
VALIDATION_PATH = BASE_DIR / "test1_rotulado.xlsx"
TEST_PATH = BASE_DIR / "test2.xlsx"

OUTPUT_DIR = BASE_DIR / "ensemble_bert_svm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "neuralmind/bert-large-portuguese-cased"

LABEL2ID = {"c1": 0, "c234": 1, "c5": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

# Divisão interna do train.xlsx
VALID_SIZE = 0.20
SPLIT_SEED = 42

# Seeds do BERTimbau
SEEDS = [42, 123, 2024, 2025]
MAX_LEN = 256
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4       # batch efetivo = 4 * 4 = 16
MAX_EPOCHS = 4
PATIENCE = 2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_GRAD_NORM = 1.0
USE_FP16 = True            # apropriado para GPU T4; BF16 não é usado aqui
NUM_WORKERS = 2

SVM_PARAM_GRID = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [2],
    "tfidf__max_df": [0.85, 1.0],
    "tfidf__sublinear_tf": [True],
    "svm__C": [0.1, 1, 3, 10],
    "svm__class_weight": [None, "balanced"],
}

# Candidatos para a fusão: peso aplicado à média dos BERTimbau.
# O restante do peso vai para o SVM calibrado.
BERT_WEIGHT_CANDIDATES = [0.50, 0.60, 0.70, 0.80, 0.90]

device = torch.device("cuda")
print("Pasta:", BASE_DIR)
print("Seeds:", SEEDS)
print("Dispositivo:", device)

Mounted at /content/drive
Pasta: /content/drive/MyDrive/epPLN/ep2
Seeds: [42, 123, 2024, 2025]
Dispositivo: cuda


## 3. Funções de pré-processamento e carregamento

A normalização segue a ideia utilizada no notebook do BERTimbau: mascara URLs, leis/decretos, números de processo e telefones e converte para minúsculas. O mesmo texto normalizado é usado pelo BERT e pelo SVM para manter a comparação consistente.

Não removemos palavras funcionais nem acentos: em uma tarefa de clareza, termos como “não” podem alterar completamente o sentido.

In [4]:
def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"https?://\S+|www\.\S+", " LINK_URL ", texto)
    texto = re.sub(
        r"(?i)\b(lei|decreto)\s*(?:nº|n°|nº\.|º|°)?\s*\d+[\d\./-]*",
        " LEI_DECRETO ",
        texto,
    )
    texto = re.sub(r"\b\d{5,}\.\d{6}/\d{4}-\d{2}\b", " NUMERO_PROCESSO ", texto)
    texto = re.sub(
        r"\(?\d{2,4}\)?\s*\d{4,5}[-\s]?\d{4}",
        " NUMERO_TELEFONE ",
        texto,
    )
    texto = re.sub(r"\s+", " ", texto).strip().lower()
    return texto


# Carregamento do conjunto de treinamento
df_train = pd.read_excel(TRAIN_PATH)

# Carregamento do conjunto destinado à avaliação final
df_test1 = pd.read_excel(VALIDATION_PATH)

required = {"resp_text", "clarity"}

for nome, dados in [
    ("train.xlsx", df_train),
    ("test1_rotulado.xlsx", df_test1)
]:
    if not required.issubset(dados.columns):
        raise ValueError(
            f"{nome} precisa conter as colunas {required}"
        )

# Remoção de registros sem rótulo
df_train = df_train.dropna(subset=["clarity"]).copy()
df_test1 = df_test1.dropna(subset=["clarity"]).copy()

# Padronização dos rótulos
for dados in [df_train, df_test1]:
    dados["clarity"] = dados["clarity"].astype(str).str.strip().str.lower()
    dados.drop(
        index=dados[~dados["clarity"].isin(LABEL2ID)].index,
        inplace=True
    )

# Normalização dos textos
df_train["text"] = df_train["resp_text"].apply(normalizar_texto)
df_test1["text"] = df_test1["resp_text"].apply(normalizar_texto)

# Conversão dos rótulos para IDs
df_train["label"] = df_train["clarity"].map(LABEL2ID).astype(int)
df_test1["label"] = df_test1["clarity"].map(LABEL2ID).astype(int)

df_train = df_train.reset_index(drop=True)
df_test1 = df_test1.reset_index(drop=True)

print("Treinamento:", df_train.shape)
print("Teste final:", df_test1.shape)

print("\nDistribuição das classes no treinamento:")
print(df_train["clarity"].value_counts())

print("\nDistribuição das classes no teste final:")
print(df_test1["clarity"].value_counts())


Treinamento: (20092, 4)
Teste final: (900, 4)

Distribuição das classes no treinamento:
clarity
c5      6892
c234    6853
c1      6347
Name: count, dtype: int64

Distribuição das classes no teste final:
clarity
c5      349
c1      347
c234    204
Name: count, dtype: int64


## 4. Separação treino/validação

A validação fica separada do treino. Ela será usada para:
- acompanhar o treinamento dos BERTimbau;
- avaliar o SVM;
- comparar os modelos isolados;
- escolher o peso da combinação.

A divisão é estratificada para preservar a proporção das classes.

In [5]:
X = df_train["text"].to_numpy()
y = df_train["label"].to_numpy()

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=VALID_SIZE,
    random_state=SPLIT_SEED,
    stratify=y
)

# Conjunto independente para avaliação final
X_test1 = df_test1["text"].to_numpy()
y_test1 = df_test1["label"].to_numpy()

print("Treino:", len(X_train), "| Validação:", len(X_val))
print("Teste final:", len(X_test1))

print("\nTreino por classe:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nValidação por classe:")
print(pd.Series(y_val).value_counts().sort_index())

print("\nTeste final por classe:")
print(pd.Series(y_test1).value_counts().sort_index())

Treino: 16073 | Validação: 4019
Teste final: 900

Treino por classe:
0    5078
1    5482
2    5513
Name: count, dtype: int64

Validação por classe:
0    1269
1    1371
2    1379
Name: count, dtype: int64

Teste final por classe:
0    347
1    204
2    349
Name: count, dtype: int64


## 5. SVM com TF-IDF + Grid Search (5-fold)

O `LinearSVC` não fornece probabilidades diretamente. Primeiro selecionamos os hiperparâmetros usando F1-Macro na validação cruzada; depois calibramos o melhor pipeline com `CalibratedClassifierCV`, para que seja possível combinar suas probabilidades com as saídas softmax do BERT.

A calibração é ajustada somente com `X_train`/`y_train`, nunca com a validação.

In [6]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        strip_accents=None,
        lowercase=False,       # o pré-processamento já converteu para minúsculas
        token_pattern=r"(?u)\b\w+\b",
        dtype=np.float32,
        sublinear_tf=True
    )),
    ("svm", LinearSVC(dual="auto", max_iter=8000, random_state=42))
])

svm_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=SVM_PARAM_GRID,
    scoring={
        "f1_macro": "f1_macro",
        "accuracy": "accuracy"
    },
    refit="f1_macro",
    cv=svm_cv,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

print("Iniciando Grid Search do SVM...")
svm_grid.fit(X_train, y_train)

print("\nMelhor F1-Macro médio na CV:", round(svm_grid.best_score_, 4))
print("Melhores parâmetros:")
for k, v in svm_grid.best_params_.items():
    print(f"  {k}: {v}")

svm_results = pd.DataFrame(svm_grid.cv_results_).sort_values("rank_test_f1_macro")
display(svm_results[[
    "mean_test_f1_macro", "std_test_f1_macro",
    "mean_test_accuracy", "mean_train_f1_macro", "params"
]].head(10))

Iniciando Grid Search do SVM...
Fitting 3 folds for each of 32 candidates, totalling 96 fits

Melhor F1-Macro médio na CV: 0.4539
Melhores parâmetros:
  svm__C: 0.1
  svm__class_weight: balanced
  tfidf__max_df: 0.85
  tfidf__min_df: 2
  tfidf__ngram_range: (1, 2)
  tfidf__sublinear_tf: True


,mean_test_f1_macro,std_test_f1_macro,mean_test_accuracy,mean_train_f1_macro,params
5,0.453875,0.005971,0.457599,0.756333,"{'svm__C': 0.1, 'svm__class_weight': 'balanced..."
7,0.453008,0.005836,0.456728,0.756260,"{'svm__C': 0.1, 'svm__class_weight': 'balanced..."
1,0.452030,0.004977,0.455609,0.753973,"{'svm__C': 0.1, 'svm__class_weight': None, 'tf..."
3,0.451712,0.004948,0.455297,0.753508,"{'svm__C': 0.1, 'svm__class_weight': None, 'tf..."
0,0.449759,0.004689,0.453120,0.676881,"{'svm__C': 0.1, 'svm__class_weight': None, 'tf..."
2,0.449636,0.003719,0.453058,0.676392,"{'svm__C': 0.1, 'svm__class_weight': None, 'tf..."
4,0.448799,0.004599,0.452622,0.678836,"{'svm__C': 0.1, 'svm__class_weight': 'balanced..."
6,0.448703,0.004312,0.452560,0.678368,"{'svm__C': 0.1, 'svm__class_weight': 'balanced..."
15,0.445041,0.002319,0.445965,0.918723,"{'svm__C': 1, 'svm__class_weight': 'balanced',..."
13,0.445015,0.002398,0.445965,0.918817,"{'svm__C': 1, 'svm__class_weight': 'balanced',..."


### 5.1 Calibração e avaliação do SVM na validação

A calibração converte os escores do SVM em probabilidades. A validação é usada apenas para medir o desempenho após o ajuste.

In [7]:
best_svm_pipeline = svm_grid.best_estimator_

# Compatibilidade entre versões do scikit-learn:
try:
    calibrated_svm = CalibratedClassifierCV(
        estimator=best_svm_pipeline, method="sigmoid", cv=3, n_jobs=-1
    )
except TypeError:
    calibrated_svm = CalibratedClassifierCV(
        base_estimator=best_svm_pipeline, method="sigmoid", cv=3, n_jobs=-1
    )

print("Ajustando calibração do SVM...")
calibrated_svm.fit(X_train, y_train)

svm_val_proba = calibrated_svm.predict_proba(X_val)
svm_val_pred = np.argmax(svm_val_proba, axis=1)

def mostrar_metricas(nome, y_true, y_pred):
    print(f"\n===== {nome} =====")
    print("Acurácia:", round(accuracy_score(y_true, y_pred), 4))
    print("F1-Macro:", round(f1_score(y_true, y_pred, average="macro"), 4))
    print(classification_report(
        y_true, y_pred,
        labels=[0, 1, 2],
        target_names=[ID2LABEL[i] for i in range(3)],
        digits=4,
        zero_division=0
    ))
    print("Matriz de confusão (linhas=real, colunas=previsto):")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2]))

mostrar_metricas("SVM TF-IDF - validação", y_val, svm_val_pred)

Ajustando calibração do SVM...

===== SVM TF-IDF - validação =====
Acurácia: 0.4533
F1-Macro: 0.4475
              precision    recall  f1-score   support

          c1     0.4764    0.4452    0.4603      1269
        c234     0.4077    0.3253    0.3619      1371
          c5     0.4664    0.5881    0.5202      1379

    accuracy                         0.4533      4019
   macro avg     0.4501    0.4529    0.4475      4019
weighted avg     0.4495    0.4533    0.4473      4019

Matriz de confusão (linhas=real, colunas=previsto):
[[565 340 364]
 [361 446 564]
 [260 308 811]]


## 6. Dataset e arquitetura BERTimbau Large

Mantemos a arquitetura customizada do EP2: BERTimbau Large, mean pooling mascarado e cabeça MLP. O mean pooling calcula a média dos vetores dos tokens válidos, ignorando padding. A cabeça usa camada intermediária, LayerNorm, GELU e Dropout.

**Nota:** `LayerNorm` é usado no lugar de `BatchNorm1d` porque tende a ser mais estável com batches pequenos e com o último batch incompleto.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            [str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )
        self.labels = None if labels is None else torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item


class BERTimbauMeanPoolingClassifier(nn.Module):
    def __init__(self, model_name, num_classes=3, dropout_rate=0.30):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.bert.gradient_checkpointing_enable()  # reduz VRAM; pode aumentar o tempo de treino
        hidden_size = self.bert.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        summed = torch.sum(token_embeddings * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        mean_pooled = summed / counts
        return self.classifier(mean_pooled)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Reprodutibilidade razoável sem impor deterministic algorithms, que podem ser lentos.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_loader(texts, labels=None, batch_size=8, shuffle=False):
    ds = TextDataset(texts, labels)
    return DataLoader(
        ds, batch_size=batch_size, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=True
    )


def autocast_context():
    return torch.cuda.amp.autocast(enabled=USE_FP16)


def predict_bert_proba(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Inferência", leave=False):
            ids = batch["input_ids"].to(device, non_blocking=True)
            mask = batch["attention_mask"].to(device, non_blocking=True)
            with autocast_context():
                logits = model(ids, mask)
            probs = torch.softmax(logits.float(), dim=1)
            all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## 7. Treinar BERTimbau com uma seed e early stopping

Cada seed começa com pesos pré-treinados iguais, mas a cabeça, a ordem dos minibatches e as operações estocásticas variam. O checkpoint com melhor F1-Macro na validação é guardado.

O gradiente é acumulado para simular batch efetivo maior. O scheduler considera o número real de atualizações do otimizador. A loss é ponderada pelo tamanho do grupo de acumulação, inclusive no último grupo incompleto.

In [9]:
def treinar_bert_seed(seed, Xtr, ytr, Xva, yva, output_dir, max_epochs=MAX_EPOCHS):
    set_seed(seed)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = output_dir / f"bertimbau_large_seed_{seed}_best.pt"

    train_loader = make_loader(Xtr, ytr, TRAIN_BATCH_SIZE, shuffle=True)
    val_loader = make_loader(Xva, yva, EVAL_BATCH_SIZE, shuffle=False)

    model = BERTimbauMeanPoolingClassifier(
        MODEL_NAME, num_classes=NUM_LABELS, dropout_rate=0.30
    ).to(device)

    base_lr = 1.0e-5
    optimizer = torch.optim.AdamW([
        {"params": model.bert.parameters(), "lr": base_lr},
        {"params": model.classifier.parameters(), "lr": base_lr * 2.0},
    ], weight_decay=WEIGHT_DECAY)

    updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    total_steps = updates_per_epoch * max_epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)
    criterion = nn.CrossEntropyLoss()

    best_f1 = -1.0
    best_epoch = 0
    epochs_without_improvement = 0
    history = []

    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        n_seen = 0

        progress = tqdm(train_loader, desc=f"Seed {seed} | época {epoch+1}/{max_epochs}")
        for step, batch in enumerate(progress):
            ids = batch["input_ids"].to(device, non_blocking=True)
            mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            # Corrige a escala do último grupo incompleto de acumulação.
            group_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
            group_size = min(GRAD_ACCUM_STEPS, len(train_loader) - group_start)

            with autocast_context():
                logits = model(ids, mask)
                loss = criterion(logits, labels) / group_size

            scaler.scale(loss).backward()
            running_loss += loss.item() * group_size * len(labels)
            n_seen += len(labels)

            is_update_step = (
                (step + 1) % GRAD_ACCUM_STEPS == 0
                or (step + 1) == len(train_loader)
            )
            if is_update_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        val_proba = predict_bert_proba(model, val_loader)
        val_pred = np.argmax(val_proba, axis=1)
        val_f1 = f1_score(yva, val_pred, average="macro")
        val_acc = accuracy_score(yva, val_pred)
        epoch_loss = running_loss / max(n_seen, 1)
        history.append({
            "seed": seed, "epoch": epoch + 1,
            "train_loss": epoch_loss, "val_f1_macro": val_f1,
            "val_accuracy": val_acc
        })

        print(
            f"Seed {seed} | época {epoch+1}: "
            f"loss={epoch_loss:.4f} | val F1-Macro={val_f1:.4f} | "
            f"val acc={val_acc:.4f}"
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
            print("  -> melhor checkpoint salvo.")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print("  -> early stopping.")
                break

    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_f1": best_f1,
        "checkpoint": str(checkpoint_path),
        "history": history
    }

## 8. Treinar os modelos multi-seed e gerar probabilidades de validação

Os modelos são treinados sequencialmente para não manter três BERT Large simultaneamente na VRAM. Depois, cada checkpoint é recarregado individualmente para inferência. A média das probabilidades (não dos rótulos) forma o ensemble de BERTimbau.

In [10]:
bert_runs = []
bert_val_probabilities = []

for seed in SEEDS:
    print("\n" + "=" * 70)
    print(f"TREINAMENTO BERTIMBAU LARGE - SEED {seed}")
    print("=" * 70)

    run = treinar_bert_seed(
        seed, X_train, y_train, X_val, y_val,
        OUTPUT_DIR / "bert_validation"
    )
    bert_runs.append(run)

    model = BERTimbauMeanPoolingClassifier(
        MODEL_NAME, num_classes=NUM_LABELS, dropout_rate=0.30
    ).to(device)
    state = torch.load(run["checkpoint"], map_location=device)
    model.load_state_dict(state)
    val_loader = make_loader(X_val, None, EVAL_BATCH_SIZE, shuffle=False)
    probs = predict_bert_proba(model, val_loader)
    bert_val_probabilities.append(probs)

    del model, val_loader
    gc.collect()
    torch.cuda.empty_cache()

bert_val_proba = np.mean(np.stack(bert_val_probabilities, axis=0), axis=0)
bert_val_pred = np.argmax(bert_val_proba, axis=1)

print("\nÉpocas escolhidas por seed:")
display(pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in bert_runs]))
mostrar_metricas("BERTimbau Large - média multi-seed - validação", y_val, bert_val_pred)


TREINAMENTO BERTIMBAU LARGE - SEED 42


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.34GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

... (70 outputs left out)

Inferência:   0%|          | 0/503 [00:00<?, ?it/s]


Épocas escolhidas por seed:


,seed,best_epoch,best_val_f1,checkpoint
0,42,2,0.454382,/content/drive/MyDrive/epPLN/ep2/ensemble_bert...
1,123,4,0.452056,/content/drive/MyDrive/epPLN/ep2/ensemble_bert...
2,2024,3,0.460759,/content/drive/MyDrive/epPLN/ep2/ensemble_bert...
3,2025,3,0.449325,/content/drive/MyDrive/epPLN/ep2/ensemble_bert...



===== BERTimbau Large - média multi-seed - validação =====
Acurácia: 0.4573
F1-Macro: 0.4574
              precision    recall  f1-score   support

          c1     0.4622    0.4965    0.4787      1269
        c234     0.3950    0.3815    0.3881      1371
          c5     0.5143    0.4967    0.5053      1379

    accuracy                         0.4573      4019
   macro avg     0.4572    0.4582    0.4574      4019
weighted avg     0.4572    0.4573    0.4570      4019

Matriz de confusão (linhas=real, colunas=previsto):
[[630 391 248]
 [449 523 399]
 [284 410 685]]


## 9. Ensemble BERTimbau + SVM: seleção do peso na validação

A fusão é feita por média ponderada das probabilidades:

`P_final = alpha * P_BERT + (1 - alpha) * P_SVM`

`P_BERT` já é a média das probabilidades dos modelos multi-seed. O peso é escolhido pelo F1-Macro na validação. Como esse mesmo conjunto participou da escolha, não interprete o melhor resultado de peso como avaliação independente; ele é uma escolha de configuração. O teste oficial, se houver gabarito separado, é que deve sustentar a comparação final.

In [11]:
fusion_rows = []
best_alpha = None
best_fusion_f1 = -1.0
best_fusion_proba = None

for alpha in BERT_WEIGHT_CANDIDATES:
    fused = alpha * bert_val_proba + (1.0 - alpha) * svm_val_proba
    pred = np.argmax(fused, axis=1)
    score = f1_score(y_val, pred, average="macro")
    fusion_rows.append({
        "peso_bert": alpha,
        "peso_svm": 1.0 - alpha,
        "f1_macro": score,
        "accuracy": accuracy_score(y_val, pred)
    })
    if score > best_fusion_f1:
        best_fusion_f1 = score
        best_alpha = alpha
        best_fusion_proba = fused

fusion_table = pd.DataFrame(fusion_rows).sort_values("f1_macro", ascending=False)
display(fusion_table)

print("Peso BERT escolhido:", best_alpha)
print("Peso SVM escolhido:", round(1.0 - best_alpha, 2))
fusion_val_pred = np.argmax(best_fusion_proba, axis=1)
mostrar_metricas("ENSEMBLE BERTIMBAU + SVM - validação", y_val, fusion_val_pred)

,peso_bert,peso_svm,f1_macro,accuracy
2,0.7,0.3,0.459526,0.460562
0,0.5,0.5,0.459154,0.461060
4,0.9,0.1,0.456896,0.457079
1,0.6,0.4,0.456784,0.458323
3,0.8,0.2,0.456503,0.457079


Peso BERT escolhido: 0.7
Peso SVM escolhido: 0.3

===== ENSEMBLE BERTIMBAU + SVM - validação =====
Acurácia: 0.4606
F1-Macro: 0.4595
              precision    recall  f1-score   support

          c1     0.4676    0.4941    0.4805      1269
        c234     0.3987    0.3676    0.3825      1371
          c5     0.5092    0.5221    0.5156      1379

    accuracy                         0.4606      4019
   macro avg     0.4585    0.4613    0.4595      4019
weighted avg     0.4584    0.4606    0.4591      4019

Matriz de confusão (linhas=real, colunas=previsto):
[[627 375 267]
 [440 504 427]
 [274 385 720]]


## 10. Comparação dos modelos na validação

Compare F1-Macro, acurácia e resultados por classe. Um ensemble só deve ser mantido se trouxer ganho consistente, especialmente na classe mais difícil, sem degradar demais as demais.

In [12]:
comparacao = []
for nome, pred in [
    ("SVM TF-IDF", svm_val_pred),
    ("BERTimbau multi-seed", bert_val_pred),
    ("Ensemble BERT + SVM", fusion_val_pred),
]:
    comparacao.append({
        "modelo": nome,
        "accuracy": accuracy_score(y_val, pred),
        "f1_macro": f1_score(y_val, pred, average="macro"),
        "precision_macro": precision_score(y_val, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_val, pred, average="macro", zero_division=0),
    })

comparacao_df = pd.DataFrame(comparacao).sort_values("f1_macro", ascending=False)
display(comparacao_df)

,modelo,accuracy,f1_macro,precision_macro,recall_macro
2,Ensemble BERT + SVM,0.460562,0.459526,0.458496,0.461274
1,BERTimbau multi-seed,0.457328,0.457399,0.457165,0.458221
0,SVM TF-IDF,0.453347,0.447452,0.450143,0.452883


## 11. Treinamento final: reajustar no conjunto completo

Depois da seleção, os modelos finais são treinados com todos os exemplos rotulados de `train.xlsx`.

Para os BERTimbau, usamos a mediana do número de épocas em que cada seed obteve seu melhor F1 na validação. Isso evita usar a validação durante o ajuste final e mantém uma duração de treinamento informada pelo experimento anterior.

O SVM é novamente otimizado por CV no conjunto completo e calibrado. O peso `best_alpha` permanece o escolhido na etapa anterior.

In [13]:
from sklearn.base import clone

# Reutiliza a configuração selecionada na seção 5
final_svm_pipeline = clone(svm_grid.best_estimator_)

# Calibra o SVM utilizando todos os dados rotulados
try:
    final_calibrated_svm = CalibratedClassifierCV(
        estimator=final_svm_pipeline,
        method="sigmoid",
        cv=3,
        n_jobs=-1
    )
except TypeError:
    final_calibrated_svm = CalibratedClassifierCV(
        base_estimator=final_svm_pipeline,
        method="sigmoid",
        cv=3,
        n_jobs=-1
    )

final_calibrated_svm.fit(X, y)

print("SVM final ajustado.")
print("Parâmetros utilizados:", svm_grid.best_params_)

SVM final ajustado.
Parâmetros utilizados: {'svm__C': 0.1, 'svm__class_weight': 'balanced', 'tfidf__max_df': 0.85, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}


## 12. Ajustar os BERTimbau finais no dataset completo

Cada seed é treinada por `FINAL_EPOCHS`. Como não existe validação nesta etapa, salvamos o checkpoint da última época. Os dados completos são usados somente depois que arquitetura, hiperparâmetros, quantidade de seeds e peso da fusão já foram definidos.

In [ ]:
# Hiperparâmetros do treino final
FINAL_EPOCHS = 4  # Altere para a quantidade de épocas desejada (ex: 3, 4 ou 5)

final_checkpoints = []

for seed in SEEDS:
    path = OUTPUT_DIR / "bert_final" / f"bertimbau_large_seed_{seed}_final.pt"

    saved = treinar_bert_final(
        seed,
        X,
        y,
        FINAL_EPOCHS,
        path
    )

    final_checkpoints.append(saved)

print("Checkpoints finais:")
for p in final_checkpoints:
    print(p)
def treinar_bert_final(seed, texts, labels, epochs, checkpoint_path):
    set_seed(seed)
    train_loader = make_loader(texts, labels, TRAIN_BATCH_SIZE, shuffle=True)

    model = BERTimbauMeanPoolingClassifier(
        MODEL_NAME, num_classes=NUM_LABELS, dropout_rate=0.30
    ).to(device)

    base_lr = 1.0e-5
    optimizer = torch.optim.AdamW([
        {"params": model.bert.parameters(), "lr": base_lr},
        {"params": model.classifier.parameters(), "lr": base_lr * 2.0},
    ], weight_decay=WEIGHT_DECAY)

    updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    total_steps = updates_per_epoch * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss, seen = 0.0, 0

        for step, batch in enumerate(tqdm(train_loader, desc=f"Final seed {seed} | época {epoch+1}/{epochs}")):
            ids = batch["input_ids"].to(device, non_blocking=True)
            mask = batch["attention_mask"].to(device, non_blocking=True)
            labels_batch = batch["labels"].to(device, non_blocking=True)

            group_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
            group_size = min(GRAD_ACCUM_STEPS, len(train_loader) - group_start)

            with autocast_context():
                logits = model(ids, mask)
                loss = criterion(logits, labels_batch) / group_size

            scaler.scale(loss).backward()
            total_loss += loss.item() * group_size * len(labels_batch)
            seen += len(labels_batch)

            if ((step + 1) % GRAD_ACCUM_STEPS == 0) or ((step + 1) == len(train_loader)):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        print(f"Seed {seed} | época {epoch+1}: loss={total_loss/max(seen,1):.4f}")

    checkpoint_path = Path(checkpoint_path)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), checkpoint_path)

    del model, optimizer, scheduler, scaler, train_loader
    gc.collect()
    torch.cuda.empty_cache()
    return str(checkpoint_path)


final_checkpoints = []

for seed in SEEDS:
    path = OUTPUT_DIR / "bert_final" / f"bertimbau_large_seed_{seed}_final.pt"

    saved = treinar_bert_final(
        seed,
        X,
        y,
        FINAL_EPOCHS,
        path
    )

    final_checkpoints.append(saved)

print("Checkpoints finais:")
for p in final_checkpoints:
    print(p)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-large-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Final seed 42 | época 1/4:   0%|          | 0/5023 [00:00<?, ?it/s]

Seed 42 | época 1: loss=1.0878


Final seed 42 | época 2/4:   0%|          | 0/5023 [00:00<?, ?it/s]

Seed 42 | época 2: loss=1.0131


... (27 outputs left out)

Seed 2025 | época 1: loss=1.0842


Final seed 2025 | época 2/4:   0%|          | 0/5023 [00:00<?, ?it/s]

Seed 2025 | época 2: loss=1.0245


Final seed 2025 | época 3/4:   0%|          | 0/5023 [00:00<?, ?it/s]

## 13. Previsões finais em `test2.xlsx` e exportação

O arquivo de teste precisa ter a coluna `resp_text`. As probabilidades dos BERTimbau são calculadas individualmente e depois promediadas. Em seguida, fazemos a fusão com as probabilidades calibradas do SVM.

In [ ]:
# ==========================================
# AVALIAÇÃO FINAL NO TEST1_ROTULADO.XLSX
# ==========================================

print("Avaliando o ensemble no test1_rotulado.xlsx...")

test1_loader = make_loader(
    X_test1, None, EVAL_BATCH_SIZE, shuffle=False
)

test1_bert_probs_each = []

for seed, checkpoint in zip(SEEDS, final_checkpoints):
    print(f"Carregando BERTimbau final - seed {seed}...")

    model = BERTimbauMeanPoolingClassifier(
        MODEL_NAME,
        num_classes=NUM_LABELS,
        dropout_rate=0.30
    ).to(device)

    model.load_state_dict(
        torch.load(checkpoint, map_location=device)
    )

    probs = predict_bert_proba(model, test1_loader)
    test1_bert_probs_each.append(probs)

    del model
    gc.collect()
    torch.cuda.empty_cache()

# Média das probabilidades dos BERTimbau
test1_bert_proba = np.mean(
    np.stack(test1_bert_probs_each, axis=0),
    axis=0
)

# Probabilidades do SVM
test1_svm_proba = final_calibrated_svm.predict_proba(X_test1)

# Fusão das probabilidades
test1_ensemble_proba = (
    best_alpha * test1_bert_proba
    + (1.0 - best_alpha) * test1_svm_proba
)

test1_predictions = np.argmax(test1_ensemble_proba, axis=1)

# Métricas finais
mostrar_metricas(
    "ENSEMBLE BERTIMBAU + SVM - TESTE FINAL",
    y_test1,
    test1_predictions
)

# Salvar resultados da avaliação
test1_results = df_test1.copy()
test1_results["predicted_clarity"] = [
    ID2LABEL[int(i)] for i in test1_predictions
]

for class_id, class_name in ID2LABEL.items():
    test1_results[f"prob_{class_name}"] = (
        test1_ensemble_proba[:, class_id]
    )

test1_output_path = OUTPUT_DIR / "avaliacao_final_test1.xlsx"
test1_results.to_excel(test1_output_path, index=False)

print("Avaliação final salva em:", test1_output_path)

if not TEST_PATH.exists():
    print(f"Não encontrei {TEST_PATH}. Pule esta célula ou ajuste TEST_PATH.")
else:
    test_df = pd.read_excel(TEST_PATH)
    if "resp_text" not in test_df.columns:
        raise ValueError("test2.xlsx precisa conter a coluna resp_text.")

    X_test = test_df["resp_text"].apply(normalizar_texto).to_numpy()

    test_loader = make_loader(X_test, None, EVAL_BATCH_SIZE, shuffle=False)
    final_bert_probs_each = []

    for seed, checkpoint in zip(SEEDS, final_checkpoints):
        print(f"Carregando modelo final da seed {seed}...")
        model = BERTimbauMeanPoolingClassifier(
            MODEL_NAME, num_classes=NUM_LABELS, dropout_rate=0.30
        ).to(device)
        model.load_state_dict(torch.load(checkpoint, map_location=device))
        probs = predict_bert_proba(model, test_loader)
        final_bert_probs_each.append(probs)

        del model
        gc.collect()
        torch.cuda.empty_cache()

    final_bert_proba = np.mean(np.stack(final_bert_probs_each, axis=0), axis=0)
    final_svm_proba = final_calibrated_svm.predict_proba(X_test)

    final_ensemble_proba = (
        best_alpha * final_bert_proba
        + (1.0 - best_alpha) * final_svm_proba
    )
    final_predictions = np.argmax(final_ensemble_proba, axis=1)

    output_df = test_df.copy()
    output_df["clarity"] = [ID2LABEL[int(i)] for i in final_predictions]
    for class_id, class_name in ID2LABEL.items():
        output_df[f"prob_{class_name}"] = final_ensemble_proba[:, class_id]

    output_path = OUTPUT_DIR / "ensemble_bertimbau_svm_test2.xlsx"
    output_df.to_excel(output_path, index=False)
    print("Arquivo gerado:", output_path)
    display(output_df.head())

## 14. Salvar configurações e resultados para documentação

Guarde as métricas, hiperparâmetros, pesos e seeds para que o experimento possa ser reproduzido e descrito no relatório.

In [ ]:
experiment_summary = {
    "model_name": MODEL_NAME,
    "seeds": SEEDS,
    "max_length": MAX_LEN,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "final_epochs": FINAL_EPOCHS,
    "svm_best_params_validation": {k: str(v) for k, v in svm_grid.best_params_.items()},
    "svm_best_cv_f1_macro": float(svm_grid.best_score_),
    "bert_weight": float(best_alpha),
    "svm_weight": float(1.0 - best_alpha),
    "validation_metrics": comparacao_df.to_dict(orient="records"),
    "bert_seed_runs": [
        {k: v for k, v in r.items() if k != "history"} for r in bert_runs
    ]
    "train_size": len(X),
    "validation_size": len(X_val),
    "final_test_size": len(X_test1),
    "final_test_metrics": {
        "accuracy": float(accuracy_score(y_test1, test1_predictions)),
        "f1_macro": float(f1_score(
            y_test1, test1_predictions, average="macro"
        ))
    },
}

summary_path = OUTPUT_DIR / "experiment_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(experiment_summary, f, ensure_ascii=False, indent=2)

history_df = pd.DataFrame([
    row for run in bert_runs for row in run["history"]
])
history_path = OUTPUT_DIR / "bert_training_history.csv"
history_df.to_csv(history_path, index=False)

print("Resumo:", summary_path)
print("Histórico:", history_path)

---

# Observações metodológicas importantes

1. **Seed ensemble:** a média de probabilidades dos BERTimbau reduz a variabilidade entre inicializações, mas não garante melhora. Se os modelos cometerem erros muito parecidos, o ganho pode ser pequeno.
2. **Diversidade:** o SVM fornece uma representação TF-IDF lexical, diferente das representações contextuais do BERT. Essa diferença pode complementar os erros do Transformer.
3. **Métrica:** a seleção usa F1-Macro, não apenas acurácia, para evitar favorecer a classe mais frequente.
4. **Grid Search:** a busca do SVM usa 5-fold CV. Fazer CV completa para cada configuração do BERTimbau Large seria extremamente caro no Colab; por isso o BERT usa uma divisão estratificada de validação e early stopping.
5. **Peso do ensemble:** a busca de `alpha` é feita na validação e pode se ajustar ao acaso desse conjunto. Para uma avaliação acadêmica rigorosa, o ideal é reservar um teste rotulado intocado ou usar validação aninhada.
6. **Calibração:** a média de probabilidades só é razoável se as probabilidades estiverem minimamente calibradas. A calibração sigmoid do SVM ajuda, mas não garante calibração perfeita.
7. **Comparação justa:** compare os modelos na mesma divisão e com o mesmo pré-processamento. Não compare a validação deste notebook diretamente com números obtidos em divisões diferentes.
8. **Interpretabilidade:** ELI5 `show_weights` é apropriado para o SVM linear/TF-IDF. Para o BERT, use explicações locais como LIME, deixando claro que são aproximações.


##15 — Acurácia no conjunto de treino

In [ ]:
# Avaliação do ensemble final em todo o conjunto de treino

train_loader = make_loader(X, None, EVAL_BATCH_SIZE, shuffle=False)
train_bert_probs_each = []

for seed, checkpoint in zip(SEEDS, final_checkpoints):
    print(f"Calculando previsões da seed {seed}...")

    model = BERTimbauMeanPoolingClassifier(
        MODEL_NAME, num_classes=NUM_LABELS, dropout_rate=0.30
    ).to(device)

    model.load_state_dict(torch.load(checkpoint, map_location=device))
    probs = predict_bert_proba(model, train_loader)
    train_bert_probs_each.append(probs)

    del model
    gc.collect()
    torch.cuda.empty_cache()

train_bert_proba = np.mean(
    np.stack(train_bert_probs_each, axis=0), axis=0
)

train_svm_proba = final_calibrated_svm.predict_proba(X)

train_ensemble_proba = (
    best_alpha * train_bert_proba
    + (1 - best_alpha) * train_svm_proba
)

train_predictions = np.argmax(train_ensemble_proba, axis=1)

mostrar_metricas(
    "ENSEMBLE FINAL - CONJUNTO DE TREINO",
    y,
    train_predictions
)

del train_loader
gc.collect()
torch.cuda.empty_cache()

##16 — Instalar o LIME para interpretabilidade

In [ ]:
!pip -q install lime

##17 — Preparar a interpretabilidade do ensemble

In [ ]:
from lime.lime_text import LimeTextExplainer
from IPython.display import display, HTML

def ensemble_predict_proba_lime(texts):

    texts_norm = [normalizar_texto(str(t)) for t in texts]

    # Probabilidades do SVM
    svm_probs = final_calibrated_svm.predict_proba(texts_norm)

    # Probabilidades dos BERTimbau
    loader = make_loader(
        texts_norm,
        None,
        EVAL_BATCH_SIZE,
        shuffle=False
    )

    bert_probs_each = []

    for seed, checkpoint in zip(SEEDS, final_checkpoints):

        model = BERTimbauMeanPoolingClassifier(
            MODEL_NAME,
            num_classes=NUM_LABELS,
            dropout_rate=0.30
        ).to(device)

        model.load_state_dict(
            torch.load(checkpoint, map_location=device)
        )

        probs = predict_bert_proba(model, loader)
        bert_probs_each.append(probs)

        del model
        gc.collect()
        torch.cuda.empty_cache()

    bert_probs = np.mean(
        np.stack(bert_probs_each, axis=0),
        axis=0
    )

    # Fusão dos quatro componentes
    return (
        best_alpha * bert_probs
        + (1 - best_alpha) * svm_probs
    )


lime_explainer = LimeTextExplainer(
    class_names=["c1", "c234", "c5"],
    random_state=42
)

print("LIME configurado para explicar o ensemble completo.")

## 18 — Gerar explicações
Esta célula utiliza exemplos de test2.xlsx, sem precisar conhecer os rótulos verdadeiros.

In [ ]:
NUM_EXPLANATIONS = min(3, len(X_test))

lime_results = []

for idx in range(NUM_EXPLANATIONS):

    original_text = str(test_df.iloc[idx]["resp_text"])

    predicted_class = int(final_predictions[idx])
    predicted_name = ID2LABEL[predicted_class]

    print(f"\nExemplo {idx}")
    print(f"Classe prevista: {predicted_name}")
    print(f"Texto: {original_text[:1000]}")

    explanation = lime_explainer.explain_instance(
        text_instance=original_text,
        classifier_fn=ensemble_predict_proba_lime,
        labels=[predicted_class],
        num_features=12,
        num_samples=300
    )

    display(HTML(
        explanation.as_html(labels=[predicted_class])
    ))

    for word, weight in explanation.as_list(label=predicted_class):
        lime_results.append({
            "indice": idx,
            "classe_prevista": predicted_name,
            "termo": word,
            "contribuicao": weight
        })

lime_results_df = pd.DataFrame(lime_results)

display(lime_results_df)

O que teremos ao final?
- Acurácia e métricas do ensemble no conjunto de treino.
- Comparação com os resultados de validação que o notebook já calcula.
- Classificações do conjunto de teste, sem avaliação por métricas.
- Explicações LIME para exemplos individuais do teste.
- Uma tabela com os termos que favoreceram ou desfavoreceram cada classificação.